# ACE/SWICS and SWIMS — Implementation / 구현

**Paper**: Gloeckler, G., Cain, J., Ipavich, F. M., Tums, E. O., Bedini, P., Fisk, L. A., Zurbuchen, T. H., Bochsler, P., Fischer, J., Wimmer-Schweingruber, R. F., Geiss, J., and Kallenbach, R., "Investigation of the Composition of Solar and Interstellar Matter Using Solar Wind and Pickup Ion Measurements with SWICS and SWIMS on the ACE Spacecraft," *Space Science Reviews* **86**, 497–539 (1998). DOI: 10.1023/A:1005036131689.

This notebook reproduces four core data-analysis concepts of the SWICS instrument on ACE, all using **synthetic** but physically-motivated data:

1. **ESA + TOF + total-E triple-coincidence ion identification matrix** — placing each ion in the (M, M/Q) plane via Eq. (1) of the paper and visualizing the species "matrix boxes" used by the on-board S³DPU.
2. **Freeze-in temperature from O⁷⁺/O⁶⁺ ratio** — computing coronal electron temperature from the charge-state ratio using ionization/recombination balance.
3. **Pickup-ion velocity distribution** — generating the interstellar H⁺ shell at 1 AU following the Vasyliunas–Siscoe (1976) f(w) prescription that SWICS measures.
4. **Solar-wind charge-state composition vs. solar cycle** — synthesizing the slow/fast wind contrast in O⁷⁺/O⁶⁺, C⁶⁺/C⁵⁺, and Fe-mean charge that SWICS samples across an 11-year cycle.

이 노트북은 ACE의 SWICS 기기의 네 가지 핵심 데이터 분석 개념을 모두 **합성** 그러나 물리적 근거가 있는 데이터로 재현한다:

1. **ESA + TOF + 총에너지 3중 일치 이온 식별 매트릭스** — 논문 식 (1)을 통해 각 이온을 (M, M/Q) 평면에 배치하고 보드 위 S³DPU가 사용하는 종 "매트릭스 박스"를 시각화.
2. **O⁷⁺/O⁶⁺ 비로부터 동결 온도** — 이온화/재결합 균형으로 전하 상태 비에서 코로나 전자 온도 계산.
3. **픽업 이온 속도 분포** — SWICS가 측정하는 Vasyliunas–Siscoe(1976) f(w) 처방에 따른 1 AU에서의 성간 H⁺ shell 생성.
4. **태양풍 전하 상태 조성 vs. 태양주기** — SWICS가 11년 주기 동안 표본 추출하는 slow/fast wind의 O⁷⁺/O⁶⁺, C⁶⁺/C⁵⁺, Fe 평균 전하 대비 합성.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.optimize import brentq

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

RNG = np.random.default_rng(seed=19970825)  # ACE launch date as seed

# Physical constants (SI).
AMU_KG = 1.66053906660e-27  # atomic mass unit, kg
EV_J = 1.602176634e-19      # electron volt, joule
K_B = 1.380649e-23          # Boltzmann constant, J/K

# SWICS instrument parameters from Gloeckler et al. 1998 Table II / Section 3.1.
TOF_PATH_M = 0.10           # carbon foil to SSD distance, 10 cm
U_A_KV = 30.0               # post-acceleration voltage, kV
ESA_K = 12.46               # E/Q analyzer constant (illustrative, main channel)

## Part 1: Triple-Coincidence Ion Identification (ESA + TOF + total E) / 3중 일치 이온 식별

SWICS identifies each detected ion through three measurements per event (Section 3.1, Eq. 1, p. 508):

* The **ESA** selects energy-per-charge $E/Q$ via the deflection voltage step.
* The **TOF** between start carbon foil and SSD gives the post-acceleration velocity, hence $\tau$.
* The **SSD** measures the residual energy $E_\text{meas}$.

From these three numbers the ion mass $M$, charge $Q$, and energy $E_\text{ion}$ are reconstructed:

$$M = 2(\tau/d)^2 E_\text{meas}/\alpha, \qquad Q \approx (E_\text{meas}/\alpha)/U_a, \qquad M/Q \approx 2(\tau/d)^2 U_a$$

with $d=10$ cm, $\alpha$ the SSD nuclear-defect factor, and $U_a$ the 30 kV post-acceleration. We will (a) synthesize a population of ten dominant solar-wind species, (b) propagate them through a forward model with realistic resolution noise, and (c) verify that they cluster into the expected matrix boxes in the (M, M/Q) plane.

SWICS는 이벤트당 세 가지 측정으로 이온을 식별한다(Section 3.1, 식 1, p. 508):

* **ESA**가 deflection 전압 step으로 $E/Q$를 선택.
* 시작 carbon foil과 SSD 사이의 **TOF**가 후가속 속도, 즉 $\tau$를 제공.
* **SSD**가 잔여 에너지 $E_\text{meas}$를 측정.

세 측정값으로 이온 질량 $M$, 전하 $Q$, 에너지 $E_\text{ion}$이 재구성된다. 우리는 (a) 10개의 주요 태양풍 종으로 이루어진 집단을 합성하고, (b) 현실적인 분해능 잡음과 함께 forward 모델로 전파하며, (c) 이들이 (M, M/Q) 평면의 예상 매트릭스 박스로 군집함을 검증한다.

In [ ]:
def species_table():
    """Return the canonical solar-wind ion species sampled by SWICS.

    Each entry follows the ordering of Gloeckler et al. 1998 Table VI (p. 530):
    typical solar-wind charge state, integer mass in amu, and a relative
    abundance multiplier representative of slow solar wind at 1 AU.

    Returns:
        list of dict: each entry contains keys 'name', 'M' (amu), 'Q'
        (elementary charges), 'rel_abundance' (dimensionless), and the
        per-species TOF reference 'tau_ref_ns' from Table VI at
        V_sw=440 km/s and U_a=30 kV.
    """
    return [
        {"name": "H+",   "M": 1.0,  "Q": 1.0,  "rel_abundance": 1.0e3, "tau_ref_ns": 38.5},
        {"name": "He2+", "M": 4.0,  "Q": 2.0,  "rel_abundance": 4.0e1, "tau_ref_ns": 53.1},
        {"name": "C6+",  "M": 12.0, "Q": 6.0,  "rel_abundance": 2.0e-1, "tau_ref_ns": 52.6},
        {"name": "N7+",  "M": 14.0, "Q": 7.0,  "rel_abundance": 5.0e-2, "tau_ref_ns": 52.6},
        {"name": "O6+",  "M": 16.0, "Q": 6.0,  "rel_abundance": 6.0e-1, "tau_ref_ns": 60.2},
        {"name": "O7+",  "M": 16.0, "Q": 7.0,  "rel_abundance": 1.5e-1, "tau_ref_ns": 56.8},
        {"name": "Ne8+", "M": 20.0, "Q": 8.0,  "rel_abundance": 6.0e-2, "tau_ref_ns": 58.1},
        {"name": "Si9+", "M": 28.0, "Q": 9.0,  "rel_abundance": 4.0e-2, "tau_ref_ns": 64.1},
        {"name": "S10+", "M": 32.0, "Q": 10.0, "rel_abundance": 1.5e-2, "tau_ref_ns": 64.9},
        {"name": "Fe11+", "M": 56.0, "Q": 11.0, "rel_abundance": 3.0e-2, "tau_ref_ns": 79.3},
    ]


def ion_velocity_km_s(eq_kev_e, m_over_q_amu_e):
    """Return ion bulk velocity from E/Q and M/Q using Eq. (1) of the paper.

    Args:
        eq_kev_e: energy-per-charge selected by the ESA, in keV per
            elementary charge.
        m_over_q_amu_e: mass-per-charge in amu per elementary charge.

    Returns:
        float or np.ndarray: ion velocity in km/s using V = 438 sqrt((E/Q)/(M/Q)).
    """
    return 438.0 * np.sqrt(eq_kev_e / m_over_q_amu_e)


def synth_swics_events(species_list, n_per_species=400, sigma_tau_frac=0.02,
                       sigma_e_frac=0.08, beta=0.7, alpha=0.6,
                       v_sw_km_s=440.0):
    """Forward-model SWICS triple-coincidence events for a list of species.

    For each species we draw n_per_species events. Each event carries an
    ESA-selected E/Q (drawn from the species' kinetic energy at v_sw), a TOF
    tau_ns sampled around the Table VI reference, and an SSD residual energy
    E_meas = alpha * beta * Q * (E/Q + U_a). Gaussian noise with the supplied
    fractional widths is added to tau and E_meas to mimic the instrument
    resolution budget of Section 4 (p. 528–531).

    Args:
        species_list: list of species dicts as returned by species_table().
        n_per_species: number of synthetic events per species.
        sigma_tau_frac: fractional Gaussian timing jitter (~ Delta tau / tau).
        sigma_e_frac: fractional Gaussian SSD noise (~ Delta E / E).
        beta: foil energy-loss factor, dimensionless.
        alpha: SSD nuclear-defect factor, dimensionless.
        v_sw_km_s: bulk solar wind speed used to set E/Q for thermal core ions.

    Returns:
        dict: arrays of length n_per_species * len(species_list) with keys
        'name', 'eq_keV', 'tau_ns', 'E_meas_keV', 'M_true', 'Q_true'.
    """
    names, eqs, taus, emeas, mtrue, qtrue = [], [], [], [], [], []
    for sp in species_list:
        # Kinetic energy of a thermal-core ion at v_sw (E = 1/2 M v^2).
        ke_kev = 0.5 * sp["M"] * AMU_KG * (v_sw_km_s * 1.0e3) ** 2 / EV_J / 1.0e3
        eq_kev = ke_kev / sp["Q"]
        # Total ion energy after post-acceleration: Q*(E/Q + U_a) keV.
        e_after_post = sp["Q"] * (eq_kev + U_A_KV)
        e_after_foil = beta * e_after_post
        e_meas_mean = alpha * e_after_foil
        # Sample TOF around the Table VI reference with fractional jitter.
        tau_samples = RNG.normal(sp["tau_ref_ns"], sigma_tau_frac * sp["tau_ref_ns"], n_per_species)
        e_samples = RNG.normal(e_meas_mean, sigma_e_frac * e_meas_mean, n_per_species)
        # ESA passband resolution ~5%, model as small jitter on E/Q.
        eq_samples = RNG.normal(eq_kev, 0.025 * eq_kev, n_per_species)
        names.extend([sp["name"]] * n_per_species)
        eqs.extend(eq_samples.tolist())
        taus.extend(tau_samples.tolist())
        emeas.extend(e_samples.tolist())
        mtrue.extend([sp["M"]] * n_per_species)
        qtrue.extend([sp["Q"]] * n_per_species)
    return {
        "name": np.asarray(names),
        "eq_keV": np.asarray(eqs),
        "tau_ns": np.asarray(taus),
        "E_meas_keV": np.asarray(emeas),
        "M_true": np.asarray(mtrue),
        "Q_true": np.asarray(qtrue),
    }


def reconstruct_M_MQ(events, beta=0.7, alpha=0.6):
    """Reconstruct (M, M/Q) for each event using Eq. (1) of the paper.

    Args:
        events: dictionary of arrays returned by synth_swics_events().
        beta: foil energy-loss factor used in the calibration.
        alpha: SSD nuclear-defect factor used in the calibration.

    Returns:
        tuple: (M_amu, M_over_Q_amu_e) two arrays in amu and amu per
        elementary charge respectively.
    """
    tau_s = events["tau_ns"] * 1.0e-9
    e_after_foil_kev = events["E_meas_keV"] / alpha
    e_after_foil_j = e_after_foil_kev * 1.0e3 * EV_J
    # M [kg] = 2 (tau/d)^2 * E_after_foil; convert to amu.
    M_kg = 2.0 * (tau_s / TOF_PATH_M) ** 2 * e_after_foil_j
    M_amu = M_kg / AMU_KG
    # M/Q [amu/e] = 2 (tau/d)^2 * (U_a + E/Q) * beta; convert factor consistently.
    ua_plus_eq_j = (U_A_KV + events["eq_keV"]) * 1.0e3 * EV_J
    M_over_Q_kg_per_e = 2.0 * (tau_s / TOF_PATH_M) ** 2 * ua_plus_eq_j * beta
    M_over_Q_amu_e = M_over_Q_kg_per_e / AMU_KG
    return M_amu, M_over_Q_amu_e


species = species_table()
events = synth_swics_events(species, n_per_species=400)
M_rec, MQ_rec = reconstruct_M_MQ(events)
print(f"Generated {len(events['name'])} synthetic events across {len(species)} species")
print(f"Reconstructed M range: {M_rec.min():.1f}–{M_rec.max():.1f} amu")
print(f"Reconstructed M/Q range: {MQ_rec.min():.2f}–{MQ_rec.max():.2f} amu/e")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

colors = plt.cm.tab10(np.linspace(0, 1, len(species)))
for sp, color in zip(species, colors):
    mask = events["name"] == sp["name"]
    ax.scatter(MQ_rec[mask], M_rec[mask], s=6, alpha=0.4, color=color, label=sp["name"])
    # Draw nominal matrix box (Section 3.3, Fig. 12): 30 boxes in (M, M/Q).
    box_mq = sp["M"] / sp["Q"]
    ax.add_patch(Rectangle((box_mq * 0.85, sp["M"] * 0.7),
                           box_mq * 0.30, sp["M"] * 0.6,
                           edgecolor=color, facecolor="none", lw=1.2, alpha=0.9))

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.7, 12.0)
ax.set_ylim(0.7, 80.0)
ax.set_xlabel("Reconstructed M/Q [amu / e] / 재구성된 M/Q")
ax.set_ylabel("Reconstructed M [amu] / 재구성된 질량")
ax.set_title("SWICS triple-coincidence ID matrix (synthetic) / SWICS 3중 일치 식별 매트릭스")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper left", ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

Each species clusters in its own region of the (M, M/Q) plane, and the 30 matrix boxes (cf. Fig. 12 of the paper) cleanly separate the dominant solar-wind ions. H⁺ sits at (1, 1), He²⁺ at (4, 2), the two oxygen charge states O⁶⁺ and O⁷⁺ sit at (16, 2.67) and (16, 2.29) respectively — close enough that mass alone could not separate them, but TOF + total-E breaks the degeneracy.

각 종은 (M, M/Q) 평면에서 자신만의 영역에 군집하며, 30개 매트릭스 박스(논문 Fig. 12 참조)가 주요 태양풍 이온을 깔끔하게 분리한다. H⁺는 (1, 1), He²⁺는 (4, 2), 산소 두 전하 상태 O⁶⁺와 O⁷⁺는 각각 (16, 2.67)과 (16, 2.29)에 위치하여 — 질량만으로는 구분 불가능하지만 TOF + 총에너지가 축퇴를 깬다.

## Part 2: Freeze-In Temperature from O⁷⁺/O⁶⁺ Ratio / 동결 온도

Once the solar-wind expansion timescale exceeds the local ionization/recombination timescale, the charge state ratio is **frozen in** (Hundhausen et al. 1968). For oxygen this happens near $\sim 1.5\,R_\odot$. The ratio of charge states satisfies

$$\frac{n(\mathrm{O}^{7+})}{n(\mathrm{O}^{6+})} = \frac{C_6(T_e)}{R_7(T_e)}$$

where $C_6$ is the O⁶⁺ → O⁷⁺ ionization rate and $R_7$ is the O⁷⁺ → O⁶⁺ radiative + dielectronic recombination rate. We use a simplified Boltzmann-type fit

$$\ln \frac{n_{O^{7+}}}{n_{O^{6+}}} = -\frac{\Delta E_{67}}{k_B T_e} + \ln A_{67}$$

with $\Delta E_{67} = 138.1$ eV the O⁶⁺ ionization potential and $A_{67}$ a dimensionless atomic-physics constant calibrated to coronal-equilibrium tables. We then invert the relation: given an observed ratio, return $T_e$.

태양풍 팽창 시간이 국소적 이온화/재결합 시간을 초과하면 전하 상태 비는 **동결**된다(Hundhausen et al. 1968). 산소의 경우 이는 약 1.5 R_⊙에서 일어난다. 단순 Boltzmann형 fit으로 비를 $T_e$로 변환한다.

In [ ]:
DELTA_E_67_EV = 138.1   # O6+ ionization potential, eV (NIST atomic-energy levels)
A_67 = 1.6e3            # dimensionless prefactor, calibrated so that ratio=0.2 at 1.6 MK


def o7_o6_ratio_from_T(T_e_K, delta_e_ev=DELTA_E_67_EV, A=A_67):
    """Return the freeze-in O7+/O6+ ratio for a coronal electron temperature.

    Uses the simplified Boltzmann form
    n(O7+)/n(O6+) = A * exp(-Delta E / kT). The prefactor A absorbs the
    detailed atomic-physics ratio C_6(T) / R_7(T) and is calibrated such that
    fast solar wind (~1.2 MK) yields ~0.05 and slow wind (~1.6 MK) yields ~0.2,
    matching Zurbuchen et al. (1999) ACE/SWICS observations.

    Args:
        T_e_K: coronal electron temperature in kelvin.
        delta_e_ev: O6+ ionization potential in electronvolt.
        A: dimensionless prefactor.

    Returns:
        float or np.ndarray: predicted O7+/O6+ ratio.
    """
    return A * np.exp(-delta_e_ev * EV_J / (K_B * T_e_K))


def T_from_o7_o6(ratio, delta_e_ev=DELTA_E_67_EV, A=A_67):
    """Invert the Boltzmann relation to obtain T_e from an observed ratio.

    Args:
        ratio: observed n(O7+) / n(O6+).
        delta_e_ev: O6+ ionization potential in electronvolt.
        A: dimensionless prefactor from o7_o6_ratio_from_T.

    Returns:
        float: freeze-in coronal electron temperature in kelvin.
    """
    return -delta_e_ev * EV_J / (K_B * np.log(ratio / A))


T_grid_K = np.linspace(0.8e6, 3.0e6, 400)
ratio_grid = o7_o6_ratio_from_T(T_grid_K)

# Three benchmark cases following Zurbuchen+ 1999.
fast_ratio = 0.045
slow_ratio = 0.2
cme_ratio = 0.8
T_fast = T_from_o7_o6(fast_ratio)
T_slow = T_from_o7_o6(slow_ratio)
T_cme = T_from_o7_o6(cme_ratio)
print(f"Fast wind  : O7+/O6+ = {fast_ratio:.3f} -> T_freeze = {T_fast / 1e6:.2f} MK")
print(f"Slow wind  : O7+/O6+ = {slow_ratio:.3f} -> T_freeze = {T_slow / 1e6:.2f} MK")
print(f"CME plasma : O7+/O6+ = {cme_ratio:.3f} -> T_freeze = {T_cme / 1e6:.2f} MK")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(T_grid_K / 1e6, ratio_grid, color="black", lw=1.5,
            label="Boltzmann freeze-in model")
ax.scatter([T_fast / 1e6], [fast_ratio], color="tab:blue", s=80, zorder=5,
           label=f"fast wind ({T_fast/1e6:.2f} MK)")
ax.scatter([T_slow / 1e6], [slow_ratio], color="tab:orange", s=80, zorder=5,
           label=f"slow wind ({T_slow/1e6:.2f} MK)")
ax.scatter([T_cme / 1e6], [cme_ratio], color="tab:red", s=80, zorder=5,
           label=f"CME plasma ({T_cme/1e6:.2f} MK)")
ax.set_xlabel("Coronal electron temperature T_e [MK] / 코로나 전자 온도")
ax.set_ylabel("O7+ / O6+ frozen-in ratio / 동결 비")
ax.set_title("SWICS freeze-in thermometer / SWICS 동결 온도계")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

The model recovers the canonical Zurbuchen et al. (1999) values: fast solar wind from coronal holes freezes near 1.2 MK, slow wind from streamer belts near 1.6 MK, and CME plasma can be much hotter (>2 MK). The exponential dependence makes O⁷⁺/O⁶⁺ a sensitive thermometer at the ~10% level around 1–2 MK.

모델은 Zurbuchen et al. (1999)의 표준 값을 회복한다: coronal hole 기원 fast wind는 약 1.2 MK 부근에서, streamer belt 기원 slow wind는 1.6 MK 부근에서, CME 플라즈마는 훨씬 더 뜨겁게(>2 MK) 동결된다. 지수적 의존성으로 1–2 MK 영역에서 ~10% 정밀도의 민감 온도계가 된다.

## Part 3: Pickup-Ion Velocity Distribution at 1 AU / 픽업 이온 속도 분포

Interstellar neutral hydrogen flowing into the heliosphere is photoionized; the freshly created H⁺ is picked up by the solar-wind motional electric field $-\mathbf{V}_\text{sw} \times \mathbf{B}$, forming a ring distribution that is rapidly pitch-angle scattered into a **shell** in the solar-wind frame at speed $V_\text{sw}$. As the plasma moves outward this shell adiabatically cools, producing the Vasyliunas–Siscoe (1976) power-law distribution:

$$f(w, r) = \frac{3}{8\pi} \frac{n_n(r) \nu_\text{ion}}{V_\text{sw}^3} w^{-3/2} \Theta(1 - w),\quad w = V/V_\text{sw}$$

with the sharp cut-off at $w = 1$ (no ion in the spacecraft frame can exceed $2 V_\text{sw}$ when picked up). SWICS measures this directly thanks to its 0.5–100 keV/e dynamic range. We synthesize the 1 AU H⁺ shell, add Poisson counting noise reflecting a 12-min cadence, and overplot the analytic distribution.

성간 중성 수소가 헬리오스피어로 흘러들어와 광이온화되면 새로 만들어진 H⁺가 태양풍 운동 전기장 $-\mathbf{V}_\text{sw} \times \mathbf{B}$에 의해 픽업되어, 태양풍 좌표계에서 속도 $V_\text{sw}$의 ring 분포를 형성한 뒤 빠르게 pitch-angle scattering되어 **shell**이 된다. 플라즈마가 외부로 이동하면서 이 shell은 단열 냉각되어 Vasyliunas–Siscoe (1976) 멱법칙 분포를 만든다. SWICS는 0.5–100 keV/e 동적 범위 덕분에 이를 직접 측정한다.

In [ ]:
def vasyliunas_siscoe_f(w, n_n_cm3=0.10, nu_ion_s=7.0e-8, v_sw_km_s=400.0):
    """Return the Vasyliunas-Siscoe (1976) pickup-ion distribution function.

    f(w) = (3 / 8 pi) (n_n nu_ion / V_sw^3) w^(-3/2) for 0 < w <= 1, else 0,
    where w = V / V_sw (V being the ion speed in the spacecraft frame relative
    to the Sun, idealised as the solar-wind frame).

    Args:
        w: dimensionless speed ratio V / V_sw.
        n_n_cm3: local neutral hydrogen density in cm^-3.
        nu_ion_s: photoionization rate in s^-1 at 1 AU.
        v_sw_km_s: solar wind bulk speed in km/s.

    Returns:
        np.ndarray: phase-space density f(w) in s^3 km^-6 (proportional units).
    """
    w = np.asarray(w, dtype=np.float64)
    f = np.zeros_like(w)
    valid = (w > 1.0e-3) & (w <= 1.0)
    f[valid] = (3.0 / (8.0 * np.pi)) * (n_n_cm3 * nu_ion_s / v_sw_km_s ** 3) * w[valid] ** -1.5
    return f


def synth_swics_pickup_counts(w_edges, exposure_s=720.0, geom_factor=2e-3,
                              v_sw_km_s=400.0, **vs_kwargs):
    """Convert the V-S distribution into expected SWICS counts per w-bin.

    The number of counts in a bin (w, w + dw) is
    N = G_iso * exposure * 4 pi V_sw^3 w^2 f(w) dw,
    where G_iso is the isotropic geometrical factor 2e-3 cm^2 sr (Section 3.1).
    Poisson noise is then sampled around N to mimic 12-min counting statistics.

    Args:
        w_edges: 1-D array of w bin edges (length n+1).
        exposure_s: integration time in seconds (12 min = 720 s default).
        geom_factor: isotropic geometrical factor in cm^2 sr.
        v_sw_km_s: solar wind speed in km/s.
        **vs_kwargs: forwarded to vasyliunas_siscoe_f.

    Returns:
        tuple: (w_centers, expected_counts, observed_counts) as 1-D arrays.
    """
    w_centers = 0.5 * (w_edges[:-1] + w_edges[1:])
    dw = np.diff(w_edges)
    f_w = vasyliunas_siscoe_f(w_centers, v_sw_km_s=v_sw_km_s, **vs_kwargs)
    # phase-space density to counts: dN/dV = 4 pi V^2 f, then integrate dV = V_sw dw.
    diff_counts = geom_factor * exposure_s * 4.0 * np.pi * (v_sw_km_s ** 3) * w_centers ** 2 * f_w * dw
    # Apply a per-cm^3 -> per-(km/s)^3 unit factor (1 cm^-3 = 1e-15 km^-3).
    diff_counts *= 1.0e-15
    # Floor extremely small means at zero so Poisson sampling is well-defined.
    diff_counts = np.clip(diff_counts, 0.0, None)
    observed = RNG.poisson(np.maximum(diff_counts, 1.0e-3))
    return w_centers, diff_counts, observed


w_edges = np.linspace(0.05, 1.20, 70)
w_c, expected_counts, observed_counts = synth_swics_pickup_counts(
    w_edges, exposure_s=12 * 60.0, n_n_cm3=0.10, nu_ion_s=7.0e-8, v_sw_km_s=400.0
)
print(f"Total expected pickup-H+ counts per 12-min cycle: {expected_counts.sum():.1f}")
print(f"Total observed (Poisson realisation):              {observed_counts.sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(w_c, expected_counts, color="black", lw=1.6, label="V-S analytic, 12 min")
ax.errorbar(w_c, observed_counts, yerr=np.sqrt(observed_counts), fmt="o",
            color="tab:blue", markersize=3, capsize=2, alpha=0.7,
            label="synthetic SWICS counts")
ax.axvline(1.0, color="tab:red", linestyle="--", lw=1.2,
           label="cut-off at V = V_sw")
ax.set_xlabel("w = V / V_sw / 정규화 속도")
ax.set_ylabel("counts per w-bin / w 빈당 카운트")
ax.set_title("Interstellar H+ pickup ion shell at 1 AU / 1 AU 성간 H+ 픽업 이온 shell")
ax.set_yscale("log")
ax.set_ylim(0.5, max(expected_counts.max(), observed_counts.max()) * 2.0)
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

The SWICS-like spectrum exhibits the canonical $w^{-3/2}$ power law and a sharp cut-off at $w = 1$ (where the ion just keeps up with the solar wind). The Poisson realisation around the analytic curve illustrates the counting-statistic limitation in a single 12-min cycle: pickup-ion analyses typically average over multiple cycles to overcome these statistics. By inverting f(w) one can map $n_n(r)$ along the solar-wind streamline, the basis for the inner-source pickup-ion science of the late 1990s.

SWICS 같은 스펙트럼은 표준적인 $w^{-3/2}$ 멱법칙과 $w = 1$에서의 sharp cut-off(이온이 태양풍과 동일 속도에 도달하는 지점)를 보인다. 해석 곡선 주변의 Poisson realization은 단일 12분 사이클에서의 카운팅 통계 한계를 보여준다 — 픽업 이온 분석은 보통 다수 사이클을 평균한다. f(w)를 역변환하면 태양풍 streamline을 따라 $n_n(r)$을 매핑할 수 있어 1990년대 후반 inner-source 픽업 이온 과학의 기반이 되었다.

## Part 4: Solar-Wind Charge-State Composition vs. Solar Cycle / 태양주기 대비 전하 상태 조성

SWICS has now sampled more than two full 11-year solar cycles. Three diagnostics in particular vary systematically with cycle phase and stream type:

* **O⁷⁺ / O⁶⁺** — freeze-in temperature proxy.
* **C⁶⁺ / C⁵⁺** — independent freeze-in proxy with a different freeze-in radius (~1.5 R⊙).
* **⟨Q_Fe⟩** — mean iron charge state, 8–11 in slow wind, ~9 in fast wind.

We synthesize a 22-year time series (1998–2020), encoding both the 11-year solar cycle and the typical fast/slow stream alternation.

SWICS는 현재까지 11년 태양주기를 두 번 이상 표본 추출했다. 특히 세 가지 진단량이 주기 단계 및 stream 종류에 따라 체계적으로 변한다.

In [ ]:
def solar_cycle_modulation(time_year, cycle_period_yr=11.0, cycle_phase_yr=2000.5):
    """Return a smooth dimensionless modulation 0 (min) -> 1 (max) of the solar cycle.

    Args:
        time_year: array of decimal years.
        cycle_period_yr: period of the solar cycle in years.
        cycle_phase_yr: a year of solar maximum (e.g. 2000.5 = peak of cycle 23).

    Returns:
        np.ndarray: modulation in [0, 1].
    """
    return 0.5 * (1.0 - np.cos(2.0 * np.pi * (time_year - cycle_phase_yr) / cycle_period_yr))


def synth_charge_state_timeseries(year_start=1998.0, year_end=2020.0, cadence_d=27.0):
    """Generate synthetic 27-day-averaged O7+/O6+, C6+/C5+, <Q_Fe> over a solar cycle.

    Slow-wind ratios are higher and fast-wind ratios are lower; the alternation is
    modulated by a Bernoulli draw weighted by the solar cycle (slow wind dominates
    near maximum, fast-wind streams from coronal holes dominate near minimum).

    Args:
        year_start: start of the time series in decimal years.
        year_end: end of the time series in decimal years.
        cadence_d: cadence in days (27 day Carrington rotation default).

    Returns:
        dict: keys 'time_year', 'O7_O6', 'C6_C5', 'Qfe', 'is_slow' (bool array).
    """
    n = int((year_end - year_start) * 365.25 / cadence_d)
    time_year = np.linspace(year_start, year_end, n)
    cycle = solar_cycle_modulation(time_year)
    # Probability of being in slow wind increases with solar activity.
    p_slow = 0.30 + 0.55 * cycle
    is_slow = RNG.uniform(size=n) < p_slow
    o7_o6 = np.where(is_slow,
                     RNG.normal(0.20, 0.04, n),
                     RNG.normal(0.045, 0.015, n))
    c6_c5 = np.where(is_slow,
                     RNG.normal(1.6, 0.30, n),
                     RNG.normal(0.60, 0.12, n))
    qfe = np.where(is_slow,
                   RNG.normal(10.6, 0.40, n),
                   RNG.normal(9.2, 0.25, n))
    return {
        "time_year": time_year,
        "O7_O6": np.clip(o7_o6, 0.005, None),
        "C6_C5": np.clip(c6_c5, 0.05, None),
        "Qfe": np.clip(qfe, 7.0, 14.0),
        "is_slow": is_slow,
        "cycle": cycle,
    }


ts = synth_charge_state_timeseries()
print(f"Synthetic SWICS timeseries: {len(ts['time_year'])} 27-day averages")
print(f"O7+/O6+ : mean = {ts['O7_O6'].mean():.3f}, slow mean = {ts['O7_O6'][ts['is_slow']].mean():.3f}, fast mean = {ts['O7_O6'][~ts['is_slow']].mean():.3f}")
print(f"C6+/C5+ : mean = {ts['C6_C5'].mean():.2f}")
print(f"<Q_Fe>  : mean = {ts['Qfe'].mean():.2f}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)

axes[0].plot(ts["time_year"], ts["cycle"], color="tab:gray", lw=1.0)
axes[0].fill_between(ts["time_year"], 0.0, ts["cycle"], color="tab:gray", alpha=0.25)
axes[0].set_ylabel("Solar cycle / 태양주기\nmodulation")
axes[0].set_ylim(0.0, 1.1)
axes[0].grid(True, alpha=0.3)
axes[0].set_title("Synthetic SWICS charge-state composition vs. solar cycle / 태양주기 대비 전하 상태 조성")

axes[1].plot(ts["time_year"], ts["O7_O6"], color="tab:blue", lw=0.6, alpha=0.4)
axes[1].plot(ts["time_year"], np.convolve(ts["O7_O6"], np.ones(13)/13, mode="same"),
             color="tab:blue", lw=1.8, label="~yearly average")
axes[1].set_ylabel("O7+ / O6+")
axes[1].set_yscale("log")
axes[1].set_ylim(0.01, 1.0)
axes[1].grid(True, which="both", alpha=0.3)
axes[1].legend(loc="upper right", fontsize=9)

axes[2].plot(ts["time_year"], ts["C6_C5"], color="tab:green", lw=0.6, alpha=0.4)
axes[2].plot(ts["time_year"], np.convolve(ts["C6_C5"], np.ones(13)/13, mode="same"),
             color="tab:green", lw=1.8)
axes[2].set_ylabel("C6+ / C5+")
axes[2].set_yscale("log")
axes[2].set_ylim(0.1, 5.0)
axes[2].grid(True, which="both", alpha=0.3)

axes[3].plot(ts["time_year"], ts["Qfe"], color="tab:purple", lw=0.6, alpha=0.4)
axes[3].plot(ts["time_year"], np.convolve(ts["Qfe"], np.ones(13)/13, mode="same"),
             color="tab:purple", lw=1.8)
axes[3].set_ylabel("<Q_Fe>")
axes[3].set_xlabel("Year / 연도")
axes[3].set_ylim(8.0, 12.5)
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

All three charge-state diagnostics swing in phase with the solar cycle: ratios are higher (and ⟨Q_Fe⟩ larger) near solar maximum when slow streamer-belt wind dominates, lower near minimum when coronal-hole fast wind dominates. This is the same qualitative pattern reported by von Steiger et al. (2000) and Zurbuchen & von Steiger (2007) using real ACE/SWICS data over cycles 23–24.

세 가지 전하 상태 진단량 모두 태양주기와 동일 위상으로 진동한다: streamer belt 기원 slow wind가 우세한 태양극대기 부근에서 비가 더 높고(그리고 ⟨Q_Fe⟩도 크고), 코로나 hole 기원 fast wind가 우세한 태양극소기 부근에서 더 낮다. 이는 von Steiger et al. (2000)과 Zurbuchen & von Steiger (2007)가 실제 ACE/SWICS cycle 23–24 데이터로 보고한 정성적 패턴과 동일하다.

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Triple-coincidence ID / 3중 일치 식별 | ESA + 30 kV post-acc + 10 cm TOF + SSD | Solar Orbiter SWA/HIS, Parker SPAN-Ion (same principle, faster electronics) |
| Mass / charge resolution / 질량·전하 분해능 | Δm/m ~ 0.16–0.47, Δ(m/q)/(m/q) ≤ 5% | SWA/HIS: m/Δm > 4; modern HMRS designs: > 100 |
| Freeze-in thermometer / 동결 온도계 | O7+/O6+ , C6+/C5+ | Same diagnostics; now combined with PFSS extrapolations |
| Pickup-ion measurement / 픽업 이온 측정 | 0.5–100 keV/e, 12-min cadence | IBEX (ENA imaging), STEREO/PLASTIC, IMAP/PUI (2025) |
| On-board classification / 보드 위 분류 | S³DPU lookup tables, 30 matrix boxes | FPGA-based ML classifiers on Solar Orbiter & PSP |
| Telemetry budget / 텔레메트리 예산 | ~510 bit/s for SWICS | Solar Orbiter SWA: ~10 kbit/s; PSP: ~3 kbit/s |

### Reproducibility note / 재현성 노트

All distributions, ratios, and time series above are **synthetic** but parameter-matched to the values reported in Gloeckler et al. (1998) Tables I–VI and follow-up SWICS papers (Zurbuchen et al. 1999, 2002; von Steiger et al. 2000). Random seeds are fixed (ACE launch date 19970825) for reproducibility. To analyse real ACE/SWICS data, the algorithms above can be applied to the SWICS Level-2 hourly products available from the ACE Science Center at Caltech (variables `O7to6`, `C6to5`, `avqFe`, and the pickup-ion phase-space density tables).

위의 모든 분포, 비, 시계열은 **합성**이지만 Gloeckler et al. (1998)의 Tables I–VI와 후속 SWICS 논문(Zurbuchen et al. 1999, 2002; von Steiger et al. 2000)이 보고한 값에 매개변수가 맞춰져 있다. 재현성을 위해 random seed가 ACE 발사일(19970825)로 고정되어 있다. 실제 ACE/SWICS 데이터를 분석하려면 위 알고리즘을 Caltech ACE Science Center에서 제공하는 SWICS Level-2 시간별 산출물(`O7to6`, `C6to5`, `avqFe` 변수, 그리고 픽업 이온 phase-space density 테이블)에 적용할 수 있다.